In [2]:
import os
import sqlalchemy
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()  
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}')

import pandas as pd 
df = pd.read_sql("SELECT * FROM Workers_data;", engine)
df

# 1. DESCRIPTIVE QUERIES 
#-----------------------
# How many workers are in the dataset?
df.shape[0]

# What is the gender distribution of workers?
df.gender.value_counts()

# What is the racial distribution of workers?
df.race.value_counts()

# What are the different education levels and how many workers fall under each?
df.education.value_counts()

# What is the most common occupation?
df.occupation.value_counts()

# What is the most common workclass?
df.workclass.value_counts()

# What is the average age of workers?
df.agg(
Average_Age=(
'age','mean')).round(
0).reset_index()

# What is the average number of hours worked per week?
df.agg(
Average_Hours_Per_Week=(
'hours-per-week','mean')).round(0).reset_index()

# What percentage of workers earn more than $50K?
df.agg(
percentage_Above_50k=(
'income_>50k', lambda x: (
x.sum()/ x.count())*100)).round(
2).reset_index()
# P = ∑x/n​ *100 This is the formula for calculating the percentage of workers earning more than $50k, 
# where ∑x is the sum of workers earning more than $50k, n is the total number of workers, 
# and the result is multiplied by 100 to express it as a percentage.;

# Which native countries are most represented?
df['native-country'].sort_values().value_counts()


# 2. DIAGONOSTIC / COMPARATIVE QUERIES
# -------------------------------
# What percentage of male workers earn more than $50K compared to female workers?
df.groupby(
'gender').agg(
percentage_above_50k=(
'income_>50k', lambda x: (
x.sum()/x.count()) * 100 )).reset_index()

# Which education level has the highest proportion of workers earning above $50K?
df.groupby(
'education').agg(
earning_above_50k=('income_>50k', 'sum')).sort_values(
by='earning_above_50k', ascending= False).reset_index()

# Which occupation has the most workers earning above $50K?
df.groupby(
'occupation').agg(
Earnings_Above_50k=(
'income_>50k','sum')).sort_values(
by= 'Earnings_Above_50k', ascending=False).reset_index()

# How does average hours per week differ across occupations?
df.groupby(
'occupation').agg(
Average_Hours_Per_Week=(
'hours-per-week','mean')).sort_values(
by= 'Average_Hours_Per_Week', ascending= False
)

# Which workclass has the highest proportion of high earners?
df.groupby(
'workclass').agg(
Earners=(
'income_>50k',lambda x: (
x.sum()/x.count())
* 100)).sort_values(
by ='Earners', ascending = False
).round(0).reset_index()

# How does marital status relate to income level?
df.groupby(
'marital-status').agg(
Percentage_Per_Income = (
'income_>50k', lambda x: (
x.sum()/x.count())
* 100
)
).sort_values(
by = 'Percentage_Per_Income', ascending =  False).round(0).reset_index()

# What is the average age of workers who earn above $50K vs those who do not?
df.groupby(
'income_>50k'
).agg(
Average_Age = (
'age', 'mean'
)
).round(0).reset_index()

# Which race group has the highest proportion of workers earning above $50K?
df.groupby(
'race'
).agg(
Proportion_Of_Workers = (
'income_>50k', lambda x:
(
x.sum()/x.count()
) * 100
)
).sort_values(
by = 'Proportion_Of_Workers', ascending = False
).round(0).reset_index()

# Do workers with more education work more hours per week?
df.groupby(
'education').agg(
Working_Hours_Per_Week = (
'hours-per-week','sum'
)
).sort_values(
by = 'Working_Hours_Per_Week', ascending = False
).reset_index()

# Which native country has the highest proportion of high earners outside the US?
Non_US_workers = df[df['native-country'] != 'United-States']
Non_US_workers.groupby('native-country').agg(
    Earners=('income_>50k', lambda x: (x.sum()/x.count())*100),
    Count=('income_>50k', 'count')
).sort_values(by='Earners', ascending=False).round(0).reset_index().head(5)

# 3. FILTERING / SLICING QUERIES
# ---------------------
# How many workers work more than 60 hours per week?
work=df[df['hours-per-week']>60]
work.groupby(
'workclass').agg(
work_hours_per_week = (
'hours-per-week','count', 
)
).sort_values(
by = 'work_hours_per_week', ascending = False
).reset_index()

# How many workers have a Doctorate or Masters degree?
degree = df[df['education'].isin(['Doctorate', 'Masters'])]
degree.groupby(
'education'
).agg(
Number_of_workers=(
'education','count'
)
).sort_values(
by='Number_of_workers', ascending=False
).reset_index()

# Who are the workers above age 60 still working in the Private sector?
aged_workers = df[(df['age'] > 60) & (df['workclass'] == 'Private')]
aged_workers.groupby(
	'workclass'
).agg(
	Workers_above_60=('workclass', 'count')
).reset_index()
print(aged_workers)

# How many female workers earn above $50K?
Female_Workers = df[df['gender']== 'Female']
Female_Workers.groupby(
'gender'
).agg(
Earners = (
'income_>50k', 'sum')
).sort_values(
by = 'Earners', ascending = False
).reset_index()

# Which workers have capital gain greater than zero?
Capital_gain = df[df['capital-gain'] > 0]
Capital_gain.groupby(
'workclass').agg(
Capital_Gain = (
'capital-gain','count'
)
).sort_values(
by = 'Capital_Gain', ascending = False
).reset_index()


       age workclass  fnlgwt     education  educational_num  \
0       67   Private  366425     Doctorate               16   
6       70   Private  216390           9th                5   
15      76   Private  316185       7th-8th                4   
32      71   Private  152307       HS-grad                9   
40      62   Private  312818     Bachelors               13   
...    ...       ...     ...           ...              ...   
43845   61   Private  176839       HS-grad                9   
43849   65   Private  195568  Some-college               10   
43875   62   Private  319582       HS-grad                9   
43882   79   Private  149912     Bachelors               13   
43928   65   Private  461715       HS-grad                9   

           marital-status         occupation   relationship   race  gender  \
0                Divorced    Exec-managerial  Not-in-family  White    Male   
6      Married-civ-spouse  Machine-op-inspct           Wife  White  Female   
15       

,workclass,Capital_Gain
0,Private,2322
1,Self-emp-not-inc,338
2,Self-emp-inc,282
3,Local-gov,254
4,Not-Stated,155
5,State-gov,138
6,Federal-gov,136
7,Without-pay,2
